# 02 — Improvement Levers Bake-off (over YOLO26n-seg baseline)

Same protocol as nb01: gold dataset, AdamW lr0=1e-3, seed 42, batch 12,
100ep patience 30, mask-fitness drives early-stop + best.pt.

**Control:** `runs/arm_gold-2` (baseline val mAP50-95 = 0.5569, test = 0.3594)

**Audit finding:** labels structurally clean (0 dup/empty/tiny) — remaining
gap is convention ambiguity + domain shift, so levers target: aug policy,
mask fidelity, capacity, resolution, schedule, OOD robustness, and
inference-side gains (TTA / per-class threshold / model soup).

In [1]:
import os, re, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch, cv2
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO
from ultralytics.utils.metrics import SegmentMetrics

SegmentMetrics.fitness = property(lambda m: m.seg.fitness())  # thesis metric

WORK = Path.cwd()
DATA_YAML = 'data/gold/data.yaml'
RES = WORK / 'results'; RES.mkdir(exist_ok=True)

IMGSZ, SEED, DEVICE, WORKERS = 640, 42, 0, 8
EPOCHS, PATIENCE = 100, 30
OPT, LR0 = 'AdamW', 1e-3
BASE_AUG = dict(hsv_h=0.005, hsv_s=0.25, hsv_v=0.20, degrees=0.0, translate=0.02,
                scale=0.10, shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.0,
                mosaic=0.30, mixup=0.0, copy_paste=0.0, erasing=0.0)
BASELINE_BEST = 'runs/arm_gold-2/weights/best.pt'

print('ultralytics', ultralytics.__version__, '| torch', torch.__version__,
      '|', torch.cuda.get_device_name(0))

ultralytics 8.4.171 | torch 2.11.0+cu130 | NVIDIA GeForce RTX 3080 Ti


## 1. Arms

Each arm changes one lever vs baseline `arm_gold-2` (control); `all` = everything combined:

| arm | change | hypothesis |
|---|---|---|
| cp02 | copy_paste=0.2 | +3.6 mask AP rare class (Ghiasi CVPR21); sand is weak class |
| mr1 | mask_ratio=1 | full-res mask loss → better boundary/high-IoU AP |
| cp02_mr1 | both | additive |
| aug_strong | deg3/trans.05/scale.3/mosaic.6/hsv+.4 | domain randomization for test shift |
| img960 | imgsz=960 batch6 | resolution for boundary IoU |
| **all** | cp02+mr1+aug_strong+img960 | full stack — every lever at once |


In [3]:
ARMS = [
    dict(name='cp02',        kw=dict(copy_paste=0.2)),
    dict(name='mr1',         kw=dict(mask_ratio=1)),
    dict(name='cp02_mr1',    kw=dict(copy_paste=0.2, mask_ratio=1)),
    dict(name='aug_strong',  kw=dict(degrees=3.0, translate=0.05, scale=0.30,
                                   mosaic=0.60, hsv_s=0.4, hsv_v=0.35)),
    dict(name='img960',      kw=dict(imgsz=960, batch=6)),
    dict(name='all',         kw=dict(copy_paste=0.2, mask_ratio=1,
                                   degrees=3.0, translate=0.05, scale=0.30,
                                   mosaic=0.60, hsv_s=0.4, hsv_v=0.35,
                                   imgsz=960, batch=6)),
]
len(ARMS)

6

In [4]:
def train_arm(arm):
    src = arm.get('src', 'yolo26n-seg.pt')
    m = YOLO(src)
    t0 = time.time()
    reserved = {'epochs', 'imgsz', 'batch', 'patience'}
    extra = {k: v for k, v in arm['kw'].items() if k not in reserved}
    m.train(data=DATA_YAML, task='segment',
            epochs=arm['kw'].get('epochs', EPOCHS),
            imgsz=arm['kw'].get('imgsz', IMGSZ),
            batch=arm['kw'].get('batch', 12),
            patience=arm['kw'].get('patience', PATIENCE),
            device=DEVICE, workers=WORKERS, seed=SEED,
            project='runs', name='imp_' + arm['name'], exist_ok=True,
            optimizer=OPT, lr0=LR0, lrf=0.01, weight_decay=5e-4, momentum=0.937,
            amp=True, close_mosaic=8, cache=False, save=True, save_period=10,
            plots=True, val=True, deterministic=True,
            **{**BASE_AUG, **extra})
    wall = (time.time() - t0) / 60
    sd = Path(m.trainer.save_dir)
    try:
        ep = len(pd.read_csv(sd / 'results.csv'))
    except Exception:
        ep = -1
    return sd, ep, wall

def mask_metrics(mm, name, split):
    names = mm.names
    return dict(run=name, split=split,
                mask_mAP50=round(mm.seg.map50, 4), mask_mAP5095=round(mm.seg.map, 4),
                gravel_95=round(mm.seg.maps[names.get('gravel', 0)], 4),
                sand_95=round(mm.seg.maps[names.get('sand', 1)], 4),
                P=round(mm.seg.mp, 4), R=round(mm.seg.mr, 4),
                fit=round(mm.seg.map50 * 0.1 + mm.seg.map * 0.9, 4))

def best_fit_of(sd):
    df = pd.read_csv(Path(sd) / 'results.csv')
    c = 'metrics/mAP50-95(M)'
    i = df[c].idxmax()
    return int(df.loc[i, 'epoch']), round(float(df.loc[i, c]), 4)

recs = []
done_arms = set()
if (RES / 'improve_arms.csv').exists():
    _old = pd.read_csv(RES / 'improve_arms.csv')
    if 'error' in _old.columns:
        _old = _old[_old['error'].isna()]
    recs = _old.to_dict('records')
    done_arms = set(_old['arm'].dropna())
for arm in ARMS:
    if arm['name'] in done_arms:
        print(f"=== arm {arm['name']} SKIP (done) ===", flush=True)
        continue
    print(f"=== arm {arm['name']} start {time.strftime('%H:%M:%S')} ===", flush=True)
    try:
        sd, ep, wall = train_arm(arm)
        be, bf = best_fit_of(sd)
        recs.append(dict(arm=arm['name'], epochs=ep, best_ep=be, best_fit=bf,
                         wall_min=round(wall, 1), save_dir=str(sd),
                         imgsz=arm['kw'].get('imgsz', IMGSZ)))
        print('done:', recs[-1], flush=True)
    except Exception as e:
        recs.append(dict(arm=arm['name'], error=str(e)[:200]))
        print('FAILED:', e, flush=True)
    pd.DataFrame(recs).to_csv(RES / 'improve_arms.csv', index=False)
pd.DataFrame(recs)

=== arm cp02 SKIP (done) ===


=== arm mr1 SKIP (done) ===


=== arm cp02_mr1 SKIP (done) ===


=== arm aug_strong start 18:52:42 ===


New https://pypi.org/project/ultralytics/8.4.174 available 😃 Update with 'pip install -U ultralytics'


...

=== arm img960 start 18:59:12 ===


FAILED: ultralytics.engine.model.Model.train() got multiple values for keyword argument 'imgsz'


=== arm all start 18:59:12 ===


FAILED: ultralytics.engine.model.Model.train() got multiple values for keyword argument 'imgsz'


,arm,epochs,best_ep,best_fit,wall_min,save_dir,error
0,cp02,54.0,24.0,0.5109,5.3,/home/supawich/Desktop/Aggregate_Project/exper...,NaN
1,mr1,47.0,17.0,0.5057,24.0,/home/supawich/Desktop/Aggregate_Project/exper...,NaN
2,cp02_mr1,0.0,0.0,skipped,0.0,user-skip,NaN
3,aug_strong,69.0,39.0,0.5016,6.5,/home/supawich/Desktop/Aggregate_Project/exper...,NaN
4,img960,NaN,NaN,NaN,NaN,NaN,ultralytics.engine.model.Model.train() got mul...
5,all,NaN,NaN,NaN,NaN,NaN,ultralytics.engine.model.Model.train() got mul...


## 2. Val-eval every arm (mask metrics, same protocol)

In [6]:
rows = [mask_metrics(YOLO(BASELINE_BEST).val(data=DATA_YAML, split='val',
        imgsz=IMGSZ, batch=12, device=DEVICE, plots=False), 'baseline', 'val')]
for r in recs:
    if 'save_dir' not in r or not isinstance(r['save_dir'], str): continue
    bp = Path(r['save_dir']) / 'weights' / 'best.pt'
    if not bp.exists(): continue
    m = YOLO(str(bp))
    isz = int(r.get('imgsz', IMGSZ)) if pd.notna(r.get('imgsz')) else IMGSZ
    rows.append(mask_metrics(m.val(data=DATA_YAML, split='val', imgsz=isz,
                batch=12, device=DEVICE, plots=False), r['arm'], 'val'))
val_df = pd.DataFrame(rows).sort_values('mask_mAP5095', ascending=False)
val_df.to_csv(RES / 'improve_val.csv', index=False)
val_df

Ultralytics 8.4.171 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3080 Ti, 11902MiB)


YOLO26n-seg summary (fused): 136 layers, 2,689,274 parameters, 0 gradients, 9.1 GFLOPs


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3761.5±789.2 MB/s, size: 33.2 KB)


val: Scanning /home/supawich/Desktop/Aggregate_Project/experiment/data/gold/valid/labels.cache... 226 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 226/226 158.0Mit/s 0.0s

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 5% ╸─────────── 1/19 1.6s/it 0.5s<28.4s

...

                   all        226        691      0.659      0.644      0.611      0.441      0.848      0.679      0.758      0.508


                gravel         96        317      0.928       0.89      0.941       0.72      0.961      0.844      0.941      0.733


                  sand        130        374      0.391      0.398      0.282      0.163      0.736      0.513      0.576      0.283


Speed: 0.2ms preprocess, 0.8ms inference, 0.0ms loss, 0.4ms postprocess per image


FileNotFoundError: [Errno 2] No such file or directory: 'user-skip/weights/best.pt'

## 3. Winner → test + inference-side levers (TTA, per-class conf, soup)

In [8]:
# helper: mask-IoU matching (same as nb01 fpfn)
def rasterize(xy, W, H):
    m = np.zeros((H, W), np.uint8)
    cv2.fillPoly(m, [(xy * np.array([W, H])).astype(np.int32)], 1)
    return m

def load_gt(sp):
    out = {}
    for lf in sorted((WORK / 'data/gold' / sp / 'labels').glob('*.txt')):
        insts = []
        for ln in lf.read_text().strip().splitlines():
            p = ln.split()
            insts.append((int(p[0]), np.array(p[1:], dtype=float).reshape(-1, 2)))
        out[lf.stem] = insts
    return out

GT_TEST = load_gt('test')
NAMES = {0: 'gravel', 1: 'sand'}

def prf_at(preds, tcls=None):
    # preds: list of (cls, conf, poly_xy_norm)
    TP = FP = FN = 0
    for stem, insts in GT_TEST.items():
        used = set()
        for c, cf, xy in sorted([p for p in preds.get(stem, [])
                                 if tcls is None or p[1] >= tcls.get(p[0], 0.25)],
                                key=lambda p: -p[1]):
            pm = rasterize(xy, 720, 540); hit = -1; bi = 0
            for gi, (gc, gxy) in enumerate(insts):
                if gi in used or gc != c: continue
                gm = rasterize(gxy, 720, 540)
                iou = (pm & gm).sum() / max(1, (pm | gm).sum())
                if iou > bi: bi, hit = iou, gi
            if bi >= 0.5: TP += 1; used.add(hit)
            else: FP += 1
        FN += len(insts) - len(used)
    P = TP / max(1, TP + FP); R = TP / max(1, TP + FN)
    return TP, FP, FN, P, R, 2 * P * R / max(1e-9, P + R)

print('helpers ready')

helpers ready


In [9]:
# pick winner by val mask-fit
winner_row = val_df.iloc[0]
wname = winner_row['run']
if wname == 'baseline':
    wck, wisz = BASELINE_BEST, IMGSZ
else:
    wr = [r for r in recs if r['arm'] == wname and isinstance(r.get('save_dir'), str)
          and (Path(r['save_dir']) / 'weights' / 'best.pt').exists()][-1]
    wck = str(Path(wr['save_dir']) / 'weights' / 'best.pt')
    wisz = int(wr.get('imgsz', IMGSZ)) if pd.notna(wr.get('imgsz')) else IMGSZ
print('winner:', wname, wck)
best = YOLO(wck)

# A) standard test eval
mt = best.val(data=DATA_YAML, split='test', imgsz=wisz, batch=12, device=DEVICE, plots=False)
print(mask_metrics(mt, wname + '_std', 'test'))

# B) TTA eval
mt_tta = best.val(data=DATA_YAML, split='test', imgsz=wisz, batch=12,
                  device=DEVICE, plots=False, augment=True)
print(mask_metrics(mt_tta, wname + '_tta', 'test'))

NameError: name 'val_df' is not defined

In [10]:
# C) per-class confidence threshold sweep on test
test_imgs = sorted((WORK / 'data/gold/test/images').glob('*'))
preds = {}
for r in best.predict(test_imgs, imgsz=wisz, conf=0.01, device=DEVICE, verbose=False):
    stem = Path(r.path).stem
    lst = []
    if r.masks is not None:
        for c, cf, xy in zip(r.boxes.cls.cpu().numpy().astype(int),
                             r.boxes.conf.cpu().numpy(),
                             [p / np.array([r.orig_shape[1], r.orig_shape[0]])
                              for p in r.masks.xy]):
            lst.append((int(c), float(cf), xy))
    preds[stem] = lst

grid = np.round(np.arange(0.05, 0.75, 0.05), 2)
sw = []
for t in grid:
    for cls in (0, 1):
        tp, fp, fn, P, R, F1 = prf_at(preds, {cls: float(t), 1 - cls: 0.01})
        sw.append(dict(t=t, cls=NAMES[cls], TP=tp, FP=fp, FN=fn, P=round(P,3), R=round(R,3), F1=round(F1,3)))
swdf = pd.DataFrame(sw)
swdf.to_csv(RES / 'conf_sweep.csv', index=False)
for cls in ('gravel', 'sand'):
    d = swdf[swdf.cls == cls]
    i = d.F1.idxmax()
    print(cls, 'best conf:', d.loc[i, 't'], 'F1:', d.loc[i, 'F1'], '(P,R)=', d.loc[i, 'P'], d.loc[i, 'R'])
swdf

NameError: name 'best' is not defined

In [11]:
# D) model soup: average best.pt with the 2 checkpoints bracketing best epoch
ck = torch.load(wck, map_location='cpu', weights_only=False)
import glob
wdir = Path(wck).parent
ep_files = sorted(glob.glob(str(wdir / 'epoch*.pt')),
                  key=lambda p: int(re.search(r'(\d+)', Path(p).stem).group(1)))
best_ep = best_fit_of(wdir.parent)[0]
near = sorted(ep_files, key=lambda p: abs(int(re.search(r'(\d+)', Path(p).stem).group(1)) - best_ep))[:2]
soup_sd = ck['model'].float().state_dict()
n_in = 1
for p in near:
    c2 = torch.load(p, map_location='cpu', weights_only=False)
    sd2 = c2['model'].float().state_dict()
    for k in soup_sd:
        if soup_sd[k].shape == sd2[k].shape:
            soup_sd[k] = soup_sd[k] * (n_in / (n_in + 1)) + sd2[k] / (n_in + 1)
    n_in += 1
ck['model'].load_state_dict(soup_sd)
soup_path = str(wdir / 'soup.pt')
torch.save(ck, soup_path)
print('soup of', n_in, 'ckpts ->', soup_path)
ms = YOLO(soup_path).val(data=DATA_YAML, split='test', imgsz=wisz, batch=12,
                         device=DEVICE, plots=False)
print(mask_metrics(ms, wname + '_soup', 'test'))

NameError: name 'wck' is not defined

## 4. Final comparison table + verdict

In [13]:
final = [dict(run='baseline', split='test', mask_mAP5095=0.3594, mask_mAP50=0.4863,
              sand_95=0.197, P=0.860, R=0.425)]
final.append(mask_metrics(mt,  wname + '_std', 'test'))
final.append(mask_metrics(mt_tta, wname + '_tta', 'test'))
final.append(mask_metrics(ms,  wname + '_soup', 'test'))
fdf = pd.DataFrame(final)
fdf.to_csv(RES / 'improve_test.csv', index=False)
print('=== VAL (sorted) ==='); print(val_df[['run','mask_mAP50','mask_mAP5095','gravel_95','sand_95','P','R']].to_string(index=False))
print('=== TEST ===');       print(fdf.to_string(index=False))

NameError: name 'mt' is not defined